## Overview: data preparation for commits regressions 

Note: the resulting tables (commits, users, projects) are rather big. Therefore, all the preliminary preparation, cleaning and merging is done on the Cloud. Only clean and short task-specific tables are used in Stata to run regressions and to prepare summary statistics. 

* Regression preparations 
    * Baseline -- commits aggregated at a country pair level, excluding users from DLK


In [ ]:
import os, sys
import pandas as pd
import sqlite3
import csv

In [ ]:
# Set path
ROOT = Path(__file__).resolve().parents[2]

DATA = ROOT / "database"
EXTRACTIONS = DATA / "extractions"
STATA = DATA / "stata"
OUTPUT = ROOT / "output"
FIGURES = OUTPUT / "figures"
TABLES = OUTPUT / "tables"

# Credentials 
credentials_path = os.environ.get("GOOGLE_APPLICATION_CREDENTIALS")

if not credentials_path:
    raise EnvironmentError(
        "GOOGLE_APPLICATION_CREDENTIALS is not set. "
        "Please set it to the path of your Google Cloud credentials JSON file."
    )

credentials_path = Path(credentials_path).expanduser().resolve()

if not credentials_path.exists():
    raise FileNotFoundError(
        f"Google Cloud credentials file not found: {credentials_path}"
    )

In [ ]:
# settings for bigquery 
dry = bigquery.QueryJobConfig(dry_run = True, use_query_cache=False) 
run = bigquery.QueryJobConfig(dry_run=False, use_query_cache=True)

In [ ]:
# example codes 
'''
# check processing costs
job = client.query(query, job_config=dry)
print("Total GB that will be processed: ", job.total_bytes_processed/1000000000)
print("Bytes billed: ", job.total_bytes_billed)

job = client.query(query, job_config=run)  #comment it out if not needed to repeat
df = job.to_dataframe()  
df.head()
'''

In [ ]:
#extraction for the baseline survey
#projects temp table (use info from first parents if forked)
#projects_commits temp table (with owner info)
#puc temp table (commits with owner and author info) 
# --> group by country_owner, country_author, year, month (baseline)
# --> if forked = 0: robustness if only direct contributions
# --> if nmloc (stayers) = 1: robustness to stayers

In [ ]:
#projects: project_id (id or if forked first parent), dummy if forked, owner_id (linked on project_id)
query='''
DROP VIEW IF EXISTS `friends-foes.commits.projects_tomerge`; 
CREATE VIEW `friends-foes.commits.projects_tomerge`
AS
SELECT DISTINCT project_id, forked_from, forked, parent_id, owner_id
FROM
    (SELECT id AS project_id, forked_from,  
        CASE
         WHEN forked_from IS NULL THEN 0
         ELSE 1
        END forked, 
        CASE
         WHEN forked_from IS NULL THEN id
         ELSE forked_from
        END parent_id     
    FROM `friends-foes.commits.projects`) pids
LEFT JOIN 
# join owner/project info on parent (rather than project) id
    (SELECT DISTINCT id, owner_id, 
    EXTRACT(YEAR FROM created_at) as project_year, EXTRACT(MONTH FROM created_at) as project_month,
    FROM `friends-foes.commits.projects`) oids
ON oids.id = pids.parent_id
'''
client.query(query, job_config=run) 

In [ ]:
# merge commits with projects to find owner location
# merge on parent_id: includes both direct (fork = 0) and indirect (fork=1) contributions 
query='''
DROP VIEW IF EXISTS `friends-foes.commits.commits_projects`;
CREATE VIEW `friends-foes.commits.commits_projects`
AS 
SELECT  c.year, c.month, c.author_id, c.project_id, commits, 
       forked, parent_id, owner_id, country_current AS country_owner, dlk AS dlk_owner,
       ru_identity AS ru_identity_owner, district_en AS district_owner, name_class AS name_class_owner
FROM 
(SELECT * FROM `friends-foes.commits.commits`) c
LEFT JOIN `friends-foes.commits.projects_tomerge` p
ON c.project_id = p.project_id
LEFT JOIN `friends-foes.user_panel.sample_locations_panel` ul
ON p.owner_id = ul.id AND c.year = ul.year 
LEFT JOIN `friends-foes.user_panel.ua_users_new` ua 
ON p.owner_id = ua.id AND c.year = ua.year 
LEFT JOIN `friends-foes.user_panel.name_class` cl 
ON p.owner_id = cl.id
# WHERE c.author_id <> p.owner_id
'''
client.query(query, job_config=run) 

In [ ]:
# merge with users to find author location
query='''
DROP TABLE IF EXISTS `friends-foes.commits.puc`; 
CREATE TABLE `friends-foes.commits.puc`
AS
SELECT DISTINCT c.year AS year, month, author_id, project_id, commits, 
       forked, parent_id, owner_id, country_owner, 
       dlk_owner, ru_identity_owner, district_owner, name_class_owner,
       country_current AS country_author, dlk AS dlk_author, 
       ru_identity AS ru_identity_author, district_en AS district_author, 
       name_class as name_class_author
FROM `friends-foes.commits.commits_projects` c
LEFT JOIN `friends-foes.user_panel.sample_locations_panel` ul
ON c.author_id = ul.id AND c.year = ul.year 
LEFT JOIN  `friends-foes.user_panel.ua_users_new` ua 
ON c.author_id = ua.id AND c.year = ua.year 
LEFT JOIN `friends-foes.user_panel.name_class` cl 
ON c.author_id = cl.id
'''
client.query(query, job_config=run) 

In [ ]:
# baseline regression
query='''
SELECT year, month, country_owner, country_author, dlk_owner, dlk_author,
       SUM(commits) AS commits_int, COUNT(commits) AS commits_ext, COUNT(DISTINCT author_id) AS n_authors, 
       COUNT(DISTINCT parent_id) AS n_projects, COUNT(DISTINCT owner_id) AS n_owners
FROM `friends-foes.commits.puc`
GROUP BY year, month, country_owner, country_author, dlk_owner, dlk_author
'''

job = client.query(query, job_config=dry)
print("Total GB that will be processed: ", job.total_bytes_processed/1000000000)
print("Bytes billed: ", job.total_bytes_billed)

In [ ]:
query='''
SELECT year, month, country_owner, country_author, dlk_owner, dlk_author,
       SUM(commits) AS commits_int, COUNT(commits) AS commits_ext, COUNT(DISTINCT author_id) AS n_authors, 
       COUNT(DISTINCT parent_id) AS n_projects, COUNT(DISTINCT owner_id) AS n_owners
FROM `friends-foes.commits.puc`
WHERE author_id <> owner_id
GROUP BY year, month, country_owner, country_author, dlk_owner, dlk_author
'''

job = client.query(query, job_config=run)  #comment it out if not needed to repeat
df = job.to_dataframe()  
df.to_csv(STATA / 'for_regressionq_bl.csv', encoding = 'utf-8', index=False)

In [ ]:
#same, but add self-commits
query='''
SELECT year, month, country_owner, country_author, dlk_owner, dlk_author,
       SUM(commits) AS commits_int, COUNT(commits) AS commits_ext, COUNT(DISTINCT author_id) AS n_authors, 
       COUNT(DISTINCT parent_id) AS n_projects, COUNT(DISTINCT owner_id) AS n_owners
FROM `friends-foes.commits.puc`
GROUP BY year, month, country_owner, country_author, dlk_owner, dlk_author
'''

job = client.query(query, job_config=run)  #comment it out if not needed to repeat
df = job.to_dataframe()  
df.to_csv(STATA / 'for_regressionq_wself.csv', encoding = 'utf-8', index=False)

In [ ]:
query='''
SELECT year, month, country_owner, country_author, ru_identity_owner, ru_identity_author, dlk_owner, dlk_author,
       SUM(commits) AS commits_int, COUNT(commits) AS commits_ext, COUNT(DISTINCT author_id) AS n_authors, 
       COUNT(DISTINCT parent_id) AS n_projects, COUNT(DISTINCT owner_id) AS n_owners
FROM `friends-foes.commits.puc`
WHERE author_id <> owner_id
GROUP BY year, month, country_owner, country_author, ru_identity_owner, ru_identity_author, dlk_owner, dlk_author
'''
job = client.query(query, job_config=run)  #comment it out if not needed to repeat
df = job.to_dataframe()  
df.to_csv(STATA / 'for_regressionq_ident.csv', encoding = 'utf-8', index=False)

In [ ]:
# with or without forks 
query='''
SELECT year, month, country_owner, country_author, forked, dlk_owner, dlk_author,
       SUM(commits) AS commits_int, COUNT(commits) AS commits_ext, COUNT(DISTINCT author_id) AS n_authors, 
       COUNT(DISTINCT parent_id) AS n_projects
FROM `friends-foes.commits.puc`
WHERE author_id <> owner_id
GROUP BY year, month, country_owner, country_author, forked, dlk_owner, dlk_author
'''
job = client.query(query, job_config=run)  #comment it out if not needed to repeat
df = job.to_dataframe()  
df.to_csv(STATA / 'for_regressionq_forks.csv', encoding = 'utf-8', index=False)

In [ ]:
# name-based identity
query='''
SELECT year, month, country_owner, country_author, ru_identity_owner, ru_identity_author, dlk_owner, dlk_author,
       name_class_owner, name_class_author,
       SUM(commits) AS commits_int, COUNT(commits) AS commits_ext, COUNT(DISTINCT author_id) AS n_authors, 
       COUNT(DISTINCT parent_id) AS n_projects, COUNT(DISTINCT owner_id) AS n_owners
FROM `friends-foes.commits.puc` c
WHERE author_id <> owner_id
GROUP BY year, month, country_owner, country_author, ru_identity_owner, ru_identity_author, dlk_owner, dlk_author, name_class_owner, name_class_author
'''
job = client.query(query, job_config=run)  #comment it out if not needed to repeat
df = job.to_dataframe()  
df.to_csv(STATA / 'for_regressionq_ident_name.csv', encoding = 'utf-8', index=False)

In [ ]:
# robustness for users who joined before 2014
query='''
SELECT year, month, country_owner, country_author, dlk_owner, dlk_author,  ru_identity_owner, ru_identity_author,
       SUM(commits) AS commits_int, COUNT(commits) AS commits_ext, COUNT(DISTINCT author_id) AS n_authors, 
       COUNT(DISTINCT parent_id) AS n_projects,     COUNT(DISTINCT owner_id) AS n_owners
FROM `friends-foes.commits.puc` 
WHERE author_id IN (SELECT DISTINCT author_id FROM `friends-foes.commits.puc` WHERE year < 2014 ) 
AND 
owner_id IN (SELECT DISTINCT owner_id  FROM `friends-foes.commits.puc` WHERE year < 2014 ) 
AND
author_id <> owner_id 
GROUP BY year, month, country_owner, country_author, dlk_owner, dlk_author,  ru_identity_owner, ru_identity_author
'''
job = client.query(query, job_config=run)  #comment it out if not needed to repeat
df = job.to_dataframe()  
df.to_csv(STATA / 'for_regressionq_2013-s.csv', encoding = 'utf-8', index=False)

In [ ]:
# movers vs stayers 
# note: it is only well defined for ua and ru users, because they have non-missing country code in 2013
query='''
CREATE VIEW `friends-foes.commits.movers`
AS
SELECT DISTINCT author_id, 
#don't count if moved in 2019 as not part of the sample 
CASE WHEN cc2013 = cc2014 AND cc2013 = cc2016 AND cc2013 = cc2019 THEN 1 ELSE 0 END stayer,
CASE WHEN  n_countries > 1 THEN 1 ELSE 0 END mover,
#CASE WHEN cc2013 = 'ru' AND (cc2014 = 'ua' OR cc2016 = 'ua' OR cc2019 = 'ua') THEN 1 ELSE 0 END mover_ru_ua,
#CASE WHEN cc2013 = 'ua' AND (cc2014 = 'ru' OR cc2016 = 'ru' OR cc2019 = 'ru') THEN 1 ELSE 0 END mover_ua_ru
FROM (SELECT DISTINCT id as author_id, cc2013, cc2014, cc2016, cc2019, 
      COUNT(DISTINCT country_current) OVER(PARTITION BY id) as n_countries 
      FROM `friends-foes.user_panel.sample_locations_panel`
      WHERE year < 2020
      ) #only until 2019
'''
job = client.query(query, job_config=run) 

In [ ]:
# robustness for users who joined before 2014 and with reported locations as of 2013
query='''
SELECT year, month, 
       CASE WHEN year < 2014 THEN c.country_author ELSE up.cc2013_author END cc2013_author, 
       CASE WHEN year < 2014 THEN c.country_owner ELSE op.cc2013_owner END cc2013_owner,
       dlk_owner, dlk_author,
       SUM(commits) AS commits_int, COUNT(commits) AS commits_ext, COUNT(DISTINCT c.author_id) AS n_authors, 
       COUNT(DISTINCT parent_id) AS n_projects, COUNT(DISTINCT c.owner_id) AS n_owners
FROM `friends-foes.commits.puc` c
LEFT JOIN (SELECT DISTINCT author_id, country_author as cc2013_author
            FROM `friends-foes.commits.puc`
            WHERE year = 2013) up
ON c.author_id = up.author_id
LEFT JOIN (SELECT DISTINCT owner_id, country_owner as cc2013_owner
            FROM `friends-foes.commits.puc`
            WHERE year = 2013) op
ON c.owner_id = op.owner_id
WHERE c.author_id IN (SELECT DISTINCT author_id FROM `friends-foes.commits.puc` m WHERE year < 2014 ) 
AND 
c.owner_id IN (SELECT DISTINCT owner_id  FROM `friends-foes.commits.puc` k WHERE year < 2014 ) 
AND c.author_id <> c.owner_id 
GROUP BY year, month, cc2013_owner, cc2013_author, dlk_owner, dlk_author
'''
job = client.query(query, job_config=run)  #comment it out if not needed to repeat
df = job.to_dataframe()  
df.to_csv(STATA / 'for_regressionq_2013.csv', encoding = 'utf-8', index=False)

In [ ]:
# stayers vs. movers with fixed locations in 2013
query='''
SELECT year, month, 
       CASE WHEN year < 2014 THEN c.country_author ELSE up.cc2013_author END cc2013_author, 
       CASE WHEN year < 2014 THEN c.country_owner ELSE op.cc2013_owner END cc2013_owner,
       dlk_owner, dlk_author,  ru_identity_owner, ru_identity_author, stayer_author, stayer_owner,
       SUM(commits) AS commits_int, COUNT(commits) AS commits_ext, COUNT(DISTINCT c.author_id) AS n_authors, 
       COUNT(DISTINCT parent_id) AS n_projects, COUNT(DISTINCT c.owner_id) AS n_owners
FROM `friends-foes.commits.puc` c
LEFT JOIN (SELECT author_id, stayer AS stayer_author FROM `friends-foes.commits.movers`) ma
ON c.author_id = ma.author_id
LEFT JOIN (SELECT author_id, stayer AS stayer_owner FROM `friends-foes.commits.movers`) mo
ON c.owner_id = mo.author_id
LEFT JOIN (SELECT DISTINCT author_id, country_author as cc2013_author
            FROM `friends-foes.commits.puc`
            WHERE year = 2013) up
ON c.author_id = up.author_id
LEFT JOIN (SELECT DISTINCT owner_id, country_owner as cc2013_owner
            FROM `friends-foes.commits.puc`
            WHERE year = 2013) op
ON c.owner_id = op.owner_id
WHERE c.author_id IN (SELECT DISTINCT author_id FROM `friends-foes.commits.puc` m WHERE year < 2014 ) 
AND 
c.owner_id IN (SELECT DISTINCT owner_id  FROM `friends-foes.commits.puc` k WHERE year < 2014 ) 
AND c.author_id <> c.owner_id 
GROUP BY year, month, cc2013_owner, cc2013_author, dlk_owner, dlk_author,  ru_identity_owner, ru_identity_author, 
         stayer_author, stayer_owner
'''
job = client.query(query, job_config=run)  #comment it out if not needed to repeat
df = job.to_dataframe()  
df.to_csv(STATA / 'for_regressionq_2013_movers.csv', encoding = 'utf-8', index=False)

In [ ]:
# heterogeneity by previous vs. old collaborations
query = '''
SELECT year, month, country_owner, country_author, dlk_owner, dlk_author,
       SUM(commits) AS commits_int, COUNT(commits) AS commits_ext, COUNT(DISTINCT author_id) AS n_authors, 
       COUNT(DISTINCT parent_id) AS n_projects, COUNT(DISTINCT owner_id) AS n_owners, new_project, 
       project_before
FROM (
  SELECT a.*, 
         CASE WHEN (ROW_NUMBER() OVER (PARTITION BY a.author_id, a.project_id ORDER BY DATE(year, month, 01))) = 1 
         THEN 1 ELSE 0 END new_project,
         CASE WHEN  (ROW_NUMBER() OVER (PARTITION BY a.author_id, a.owner_id ORDER BY DATE(year, month, 01))) = 1 
         THEN 1 ELSE 0 END new_owner, b.min_year as min_year_project, c.min_year as min_year_owner, project_before, owner_before
    FROM `friends-foes.commits.puc` a
    LEFT JOIN (SELECT MIN(year) as min_year, author_id, project_id,
               CASE WHEN MIN(year) < 2014 THEN 1 ELSE 0 END project_before
               FROM `friends-foes.commits.puc` GROUP BY author_id, project_id) b
    ON a.author_id = b.author_id AND a.project_id = b.project_id
    LEFT JOIN (SELECT MIN(year) as min_year, author_id, owner_id,
               CASE WHEN MIN(year) < 2014 THEN 1 ELSE 0 END owner_before
               FROM `friends-foes.commits.puc` GROUP BY author_id, owner_id) c
    ON a.author_id = c.author_id AND a.owner_id = c.owner_id
    WHERE country_owner <> country_author
    )
GROUP BY year, month, country_owner, country_author, dlk_owner, dlk_author, new_project, project_before'''

job = client.query(query, job_config=run)  #comment it out if not needed to repeat
df = job.to_dataframe()  
df.to_csv(STATA / 'for_regressionq_old_new.csv', encoding = 'utf-8', index=False)

In [ ]:
# heterogeneity for users with and without a company affiliation 
# run from console, because of \\N characters
query = """
DROP TABLE IF EXISTS `friends-foes.user_panel.sample_companies`;

CREATE TABLE `friends-foes.user_panel.sample_companies`
AS
SELECT DISTINCT p.id, p.year, a.company13, b.company14,  c.company16, d.company19, 
    CASE 
      WHEN p.year < 2014 AND a.company13 NOT IN ('NA', '\\N') THEN a.company13 
      WHEN p.year BETWEEN 2014 AND 2015 AND b.company14 NOT IN ('NA', '\\N') THEN b.company14
      WHEN p.year BETWEEN 2016 AND 2017 AND c.company16 NOT IN ('NA', '\\N') THEN c.company16
      WHEN p.year BETWEEN 2018 AND 2019 AND d.company19 NOT IN ('NA', '\\N') THEN d.company19
    END company_current
FROM  
    (SELECT DISTINCT id, m.year FROM `friends-foes.commits.users_sample`
    CROSS JOIN 
    (SELECT * FROM UNNEST([2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021]) AS year) m
    ) p
LEFT JOIN 
     (SELECT DISTINCT id, company as company13 FROM `friends-foes.user_panel.users2013`) a
ON p.id = a.id
LEFT JOIN 
     (SELECT DISTINCT id, company as company14 FROM `friends-foes.user_panel.users2014`) b
ON p.id = b.id
LEFT JOIN 
     (SELECT DISTINCT id, company as company16 FROM `friends-foes.user_panel.users2016`) c
ON p.id = c.id
LEFT JOIN 
     (SELECT DISTINCT id, company as company19 FROM `ghtorrentmysql1906.MySQL1906.users`) d
ON p.id = d.id
"""
client.query(query, job_config=run) 

In [ ]:
# select users with and without (current) company affiliation 
# contribute to smn affiliated with company or just a user
query='''
SELECT DISTINCT p.year, month, country_owner, country_author, dlk_owner, dlk_author, 
       CASE WHEN a.company_current IS NOT NULL THEN 1 ELSE 0 END company_owner, 
      # CASE WHEN b.company_current IS NOT NULL THEN 1 ELSE 0 END company_author,
       SUM(commits) AS commits_int, COUNT(commits) AS commits_ext, COUNT(DISTINCT author_id) AS n_authors, 
       COUNT(DISTINCT parent_id) AS n_projects, COUNT(DISTINCT owner_id) AS n_owners
FROM `friends-foes.commits.puc` p
LEFT JOIN `friends-foes.user_panel.sample_companies` a ON a.year = p.year AND a.id = p.owner_id
LEFT JOIN `friends-foes.user_panel.sample_companies` b ON b.year = p.year AND b.id = p.author_id
WHERE author_id <> owner_id
GROUP BY year, month, country_owner, country_author, company_owner, dlk_owner, dlk_author
'''

job = client.query(query, job_config=run)  #comment it out if not needed to repeat
df = job.to_dataframe()  
df.to_csv(STATA / 'for_regressionq_company.csv', encoding = 'utf-8', index=False)

In [ ]:
# count of new collaborations with a certain country; count of leaving a project with a certain country 
# last and first commits for a project 
query = '''
CREATE VIEW  `friends-foes.commits.first_last` 
AS
SELECT author_id, parent_id, owner_id, month, year,
CASE WHEN 
  ROW_NUMBER()
      OVER (PARTITION BY author_id, parent_id 
      ORDER BY DATE(year, month, 01) DESC) = 1 THEN 1 ELSE 0
END last_commit_project, 
CASE WHEN 
  ROW_NUMBER()
      OVER (PARTITION BY author_id, parent_id 
      ORDER BY DATE(year, month, 01)) = 1 THEN 1 ELSE 0
END first_commit_project, 
CASE WHEN 
  ROW_NUMBER()
      OVER (PARTITION BY author_id, owner_id 
      ORDER BY DATE(year, month, 01)) = 1 THEN 1 ELSE 0
END first_commit_owner, 
CASE WHEN 
  ROW_NUMBER()
      OVER (PARTITION BY author_id, owner_id 
      ORDER BY DATE(year, month, 01) DESC) = 1 THEN 1 ELSE 0
END last_commit_owner
FROM  `friends-foes.commits.puc` 
ORDER BY author_id, owner_id, parent_id, year, month
'''
client.query(query, job_config=run) 